# 05 · Agentic Workflow

**Goal:** let an LLM search Canadian bank filings, check which filings exist, and compare disclosures across banks or years.

In [ ]:
%pip install -q "chromadb>=1,<2" sentence-transformers openai python-dotenv numpy pandas

## 1. Setup

Choose the same embedding provider you used when creating the collection in Notebook 2. Set `CHAT_MODEL` to the model you have access to. Costs are estimates; update the two prices below for your chosen model.

In [ ]:
import json
import os
import time
from pathlib import Path

import chromadb
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display
from openai import OpenAI
from sentence_transformers import SentenceTransformer

ROOT = Path.cwd()
BASE = ROOT.parent if ROOT.name == 'notebooks' else ROOT
load_dotenv(BASE / '.env', encoding='utf-8-sig')
load_dotenv(BASE / 'notebooks' / '.env', encoding='utf-8-sig')

MODELS = BASE / 'models'
REPORTS = BASE / 'reports' / 'agentic_workflow'
REPORTS.mkdir(parents=True, exist_ok=True)

EMBEDDING_PROVIDER = 'local'  # 'local' or 'openai'
CHAT_MODEL = 'gpt-4o-mini'
LOCAL_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
OPENAI_EMBEDDING_MODEL = 'text-embedding-3-small'
LOCAL_WINDOW_TOKENS = 220  # same as Notebook 2
MAX_ITERATIONS = 8
TOP_K = 4

# Example prices in USD per million tokens. Update for your selected chat model.
INPUT_PRICE_PER_MILLION = 0.15
OUTPUT_PRICE_PER_MILLION = 0.60
EMBED_PRICE_PER_MILLION = 0.02

if EMBEDDING_PROVIDER not in ('local', 'openai'):
    raise ValueError('Choose local or openai for EMBEDDING_PROVIDER.')
if not os.getenv('OPENAI_API_KEY'):
    raise ValueError('Add OPENAI_API_KEY to the project .env file.')

llm = OpenAI(max_retries=3, timeout=60)
store = chromadb.PersistentClient(path=str(MODELS / 'chroma'))
collection_name = f'bank_filings_{EMBEDDING_PROVIDER}'
try:
    collection = store.get_collection(collection_name, embedding_function=None)
except Exception as error:
    raise ValueError(f'{collection_name} not found. Run Notebook 2 first.') from error
if collection.count() == 0:
    raise ValueError('ChromaDB collection is empty. Run Notebook 2 first.')

print(f'{collection_name}: {collection.count():,} chunks')

## 2. Search the filings

The search uses the same embedding approach as Notebook 2. Filtering by bank, year, or document type prevents mixing unrelated reports. The search function returns source details alongside each passage.

In [ ]:
def normalize(vectors):
    length = np.linalg.norm(vectors, axis=1, keepdims=True)
    return vectors / np.maximum(length, 1e-12)


if EMBEDDING_PROVIDER == 'local':
    local_model = SentenceTransformer(LOCAL_MODEL)


def embed_question(question):
    if EMBEDDING_PROVIDER == 'openai':
        response = llm.embeddings.create(model=OPENAI_EMBEDDING_MODEL, input=[question])
        return normalize(np.array([response.data[0].embedding], dtype=np.float32))[0], response.usage.total_tokens

    tokenizer = local_model.tokenizer
    size = min(LOCAL_WINDOW_TOKENS, local_model.max_seq_length - 2)
    tokens = tokenizer.encode(question, add_special_tokens=False)
    if not tokens:
        raise ValueError('Search text cannot be empty.')
    parts = [tokens[i:i + size] for i in range(0, len(tokens), size)]
    vectors = local_model.encode([tokenizer.decode(part) for part in parts], normalize_embeddings=True)
    weights = np.array([len(part) for part in parts])
    average = np.average(vectors, axis=0, weights=weights)
    return normalize(np.array([average], dtype=np.float32))[0], 0


def make_filter(**values):
    conditions = [{name: {'$eq': value}} for name, value in values.items() if value is not None]
    if not conditions:
        return None
    return conditions[0] if len(conditions) == 1 else {'$and': conditions}


def search_filings(query, bank=None, fiscal_year=None, document_type=None, section=None, k=TOP_K, quarter=None):
    """Find relevant filing passages, optionally filtering the source."""
    bank = 'BNS' if bank and bank.upper() in {'SCOTIABANK', 'BANK OF NOVA SCOTIA'} else bank
    document_type = {'annual report': 'annual_report', 'annual information form': 'AIF',
                     'aif': 'AIF', 'quarterly report': 'quarterly'}.get(
                         str(document_type).lower(), document_type) if document_type else None
    if document_type not in (None, 'annual_report', 'AIF', 'quarterly'):
        raise ValueError('document_type must be annual_report, AIF or quarterly')
    vector, tokens = embed_question(query)
    conditions = make_filter(bank_code=bank, fiscal_year=fiscal_year, document_type=document_type, quarter=quarter)
    if section:
        matching = collection.get(where=conditions, include=['metadatas'])
        names = sorted({m.get('section', '') for m in matching['metadatas']
                        if section.casefold() in str(m.get('section', '')).casefold()})
        if not names:
            return {'passages': [], 'embedding_tokens': tokens}
        section_filter = {'section': {'$in': names}}
        conditions = {'$and': [conditions, section_filter]} if conditions else section_filter
    eligible = len(collection.get(where=conditions, include=[])['ids']) if conditions else collection.count()
    if not eligible:
        return {'passages': [], 'embedding_tokens': tokens}
    result = collection.query(
        query_embeddings=[vector.tolist()],
        n_results=min(max(1, k), 8, eligible),
        where=conditions,
        include=['documents', 'metadatas', 'distances'],
    )
    passages = []
    for chunk_id, text, metadata, distance in zip(
        result['ids'][0], result['documents'][0],
        result['metadatas'][0], result['distances'][0]
    ):
        # Section names vary among the banks; apply this filter to returned results.
        if section and section.lower() not in str(metadata.get('section', '')).lower():
            continue
        passages.append({
            'chunk_id': chunk_id,
            'text': text[:5000],
            'similarity': round(1 - distance, 3),
            **metadata,
        })
    return {'passages': passages, 'embedding_tokens': tokens}


def source_title(item):
    year = item.get('fiscal_year')
    quarter = item.get('quarter')
    q = f' Q{quarter}' if quarter not in (None, '', 0, '0') else ''
    return (f"{item.get('bank_code', '?')} | {item.get('document_type', '?')} | "
            f"{year}{q} | {item.get('section') or 'Section unknown'} | "
            f"pages {item.get('page_start', '?')}-{item.get('page_end', '?')} | "
            f"{item.get('file_name') or 'filename unknown'}")

## 3. List available filings and compare sections

`get_filing_list` checks which files you actually ingested. `compare_sections` searches each requested period separately. If a year is missing, the tool reports it instead of pretending a comparison is possible.

In [ ]:
def get_filing_list(bank=None, fiscal_year=None):
    """List source documents available in the vector database."""
    bank = 'BNS' if bank and bank.upper() in {'SCOTIABANK', 'BANK OF NOVA SCOTIA'} else bank
    fields = ['filing_id', 'bank_code', 'document_type', 'fiscal_year',
              'quarter', 'file_name', 'source_url']
    rows = []
    for offset in range(0, collection.count(), 1000):
        batch = collection.get(limit=1000, offset=offset, include=['metadatas'])
        rows.extend(batch['metadatas'])
    filings = []
    seen = set()
    for item in rows:
        if bank and item.get('bank_code') != bank:
            continue
        if fiscal_year is not None and item.get('fiscal_year') != fiscal_year:
            continue
        details = {name: item.get(name) for name in fields}
        key = details.get('filing_id') or (details.get('bank_code'), details.get('document_type'),
                                           details.get('fiscal_year'), details.get('quarter'), details.get('file_name'))
        if key not in seen:
            filings.append(details)
            seen.add(key)
    return {'filings': sorted(filings, key=lambda f: (str(f['bank_code']), str(f['fiscal_year']), str(f['document_type']), str(f['quarter']))),
            'count': len(filings)}


def compare_sections(query, bank, year_a, year_b, document_type='annual_report', k=3):
    """Find matching passages in two years of one bank's reports."""
    available = get_filing_list(bank)['filings']
    years = {f.get('fiscal_year') for f in available if f.get('document_type') == document_type}
    result = {}
    total_tokens = 0
    for year in (year_a, year_b):
        if year not in years:
            result[str(year)] = {'status': 'filing not found', 'passages': []}
        else:
            found = search_filings(query, bank=bank, fiscal_year=year, document_type=document_type, k=k)
            result[str(year)] = {'status': 'found', 'passages': found['passages']}
            total_tokens += found['embedding_tokens']
    return {'bank': bank, 'query': query, 'years': result, 'embedding_tokens': total_tokens}

## 4. Define the agent's tools

OpenAI receives the tool descriptions, selects a tool, and sends arguments. Python runs the requested function. There are exactly three tools, and no arbitrary code execution.

In [ ]:
TOOLS = [
    {'type': 'function', 'function': {
        'name': 'search_filings',
        'description': 'Search Canadian bank filings for supporting passages. Optional bank, fiscal year and report type filters.',
        'parameters': {'type': 'object', 'properties': {
            'query': {'type': 'string'}, 'bank': {'type': 'string', 'enum': ['RBC', 'TD', 'BMO', 'BNS', 'CIBC']},
            'fiscal_year': {'type': 'integer'}, 'document_type': {'type': 'string', 'enum': ['annual_report', 'AIF', 'quarterly']},
            'quarter': {'type': 'integer', 'minimum': 1, 'maximum': 4},
            'section': {'type': 'string'}, 'k': {'type': 'integer', 'minimum': 1, 'maximum': 8}},
            'required': ['query'], 'additionalProperties': False}}},
    {'type': 'function', 'function': {
        'name': 'get_filing_list',
        'description': 'List which bank filings and years are actually available. Use this before cross-year comparisons.',
        'parameters': {'type': 'object', 'properties': {
            'bank': {'type': 'string', 'enum': ['RBC', 'TD', 'BMO', 'BNS', 'CIBC']},
            'fiscal_year': {'type': 'integer'}}, 'additionalProperties': False}}},
    {'type': 'function', 'function': {
        'name': 'compare_sections',
        'description': 'Retrieve evidence about the same subject from two different years for one bank. Reports missing years.',
        'parameters': {'type': 'object', 'properties': {
            'query': {'type': 'string'}, 'bank': {'type': 'string', 'enum': ['RBC', 'TD', 'BMO', 'BNS', 'CIBC']},
            'year_a': {'type': 'integer'}, 'year_b': {'type': 'integer'},
            'document_type': {'type': 'string', 'enum': ['annual_report', 'AIF', 'quarterly']}, 'k': {'type': 'integer', 'minimum': 1, 'maximum': 8}},
            'required': ['query', 'bank', 'year_a', 'year_b'], 'additionalProperties': False}}},
]

FUNCTIONS = {'search_filings': search_filings,
             'get_filing_list': get_filing_list,
             'compare_sections': compare_sections}

# For quick browsing, don't send the entire filing inventory or huge passages to the model.
def shorten_tool_result(name, result):
    if name == 'get_filing_list' and len(result['filings']) > 100:
        return {'count': result['count'], 'filings': result['filings'][:100], 'truncated': True}
    return result

## 5. Run the agent and save its actions

The agent can call several tools before writing an answer. Each action, its arguments, and its returned evidence are logged. We also count chat tokens and embedding tokens. The system prompt asks for evidence-based answers rather than invented numbers or missing filings.

In [ ]:
AGENT_INSTRUCTIONS = """You are a research assistant for Canadian bank compliance analysts.
Use the available tools to inspect actual filings before answering. Scotiabank has bank code BNS.
Use the highest fiscal year actually available for each bank when asked for the most recent report.
An empty old-year inventory prohibits a claim that a risk is new; explicitly state the comparison cannot be made. For comparisons, check which filings exist.
Answer only from tool evidence. Cite source chunk IDs and their bank, year and report type.
Do not claim a risk is newly introduced unless the relevant old and new filings were checked.
Missing a topic in retrieved passages is not proof it never appeared in the document.
If evidence or a year is missing, state the limitation explicitly.
For numerical changes, include the quoted numbers and periods; don't guess or invent them.
For ranking disclosure detail, explain your criteria and say that retrieval samples may be incomplete.
Keep responses concise. You are supporting human review, not making a compliance decision."""


def run_agent(question):
    started = time.time()
    messages = [
        {'role': 'system', 'content': AGENT_INSTRUCTIONS},
        {'role': 'user', 'content': question},
    ]
    trace = []
    input_tokens = output_tokens = embedding_tokens = 0
    final_answer = None

    for iteration in range(1, MAX_ITERATIONS + 1):
        response = llm.chat.completions.create(
            model=CHAT_MODEL, messages=messages, tools=TOOLS,
            tool_choice='auto', temperature=0, max_completion_tokens=1200,
        )
        if response.usage:
            input_tokens += response.usage.prompt_tokens
            output_tokens += response.usage.completion_tokens
        message = response.choices[0].message
        calls = message.tool_calls or []
        if not calls:
            final_answer = message.content or ''
            trace.append({'iteration': iteration, 'action': 'final_answer', 'answer': final_answer})
            break

        messages.append(message.model_dump(exclude_none=True))
        for call in calls:
            name = call.function.name
            try:
                args = json.loads(call.function.arguments)
                if name not in FUNCTIONS:
                    raise ValueError(f'Unknown tool: {name}')
                observation = FUNCTIONS[name](**args)
                embedding_tokens += observation.get('embedding_tokens', 0)
                observation = shorten_tool_result(name, observation)
            except Exception as error:
                args = call.function.arguments
                observation = {'error': str(error)}
            trace.append({'iteration': iteration, 'action': name,
                          'arguments': args, 'observation': observation,
                          'public_action_summary': message.content or f'Tool requested: {name}'})
            messages.append({'role': 'tool', 'tool_call_id': call.id,
                             'content': json.dumps(observation, ensure_ascii=False, default=str)})

    if final_answer is None:
        final_answer = 'Stopped after the maximum number of iterations. Review the tool log; no final answer was produced.'

    chat_cost = (input_tokens * INPUT_PRICE_PER_MILLION +
                 output_tokens * OUTPUT_PRICE_PER_MILLION) / 1_000_000
    embedding_cost = embedding_tokens * EMBED_PRICE_PER_MILLION / 1_000_000
    return {
        'question': question, 'answer': final_answer, 'iterations': iteration,
        'tool_calls': sum(item['action'] not in ('final_answer',) for item in trace),
        'tools_used': [item['action'] for item in trace if item['action'] != 'final_answer'],
        'input_tokens': input_tokens, 'output_tokens': output_tokens,
        'embedding_tokens': embedding_tokens,
        'estimated_cost_usd': round(chat_cost + embedding_cost, 6),
        'seconds': round(time.time() - started, 2), 'trace': trace,
    }

## 6. Try one question

Start with a single run so you can inspect the tool trace and verify that the returned chunks come from the correct filings.

In [ ]:
EXAMPLE_QUESTION = 'How does TD credit risk language compare with BMO in their most recent annual reports?'
example = run_agent(EXAMPLE_QUESTION)
print(example['answer'])
print('\nTools:', example['tools_used'])
print('Estimated cost (USD):', example['estimated_cost_usd'])

example_path = REPORTS / f'agent_example_{EMBEDDING_PROVIDER}.json'
example_path.write_text(json.dumps(example, indent=2, ensure_ascii=False, default=str), encoding='utf-8')
print('Saved trace:', example_path)

## 7. Test the five questions from Module 4

The first question asks about 2023 and 2024. If 2023 was not ingested, the agent should report that limitation instead of making up the comparison.

Set `RUN_ALL_TESTS = True`

In [ ]:
TEST_QUESTIONS = [
    'What new risks did RBC add in 2024 that were not in 2023?',
    "How does TD's credit risk language compare to BMO's in their most recent annual reports?",
    'Which of the Big Five banks has the most detailed climate risk disclosure?',
    "How did Scotiabank's provision for credit losses change over the past 4 quarters?",
    'Which banks mention AI risk or OSFI E-23 in their most recent filings?',
]

RESULTS_PATH = REPORTS / f'agent_tests_{EMBEDDING_PROVIDER}.json'
RUN_ALL_TESTS = True  # switch to True to run the five tests
RERUN = False

if RESULTS_PATH.exists() and not RERUN:
    test_results = json.loads(RESULTS_PATH.read_text(encoding='utf-8'))
    print('Loaded saved results')
elif RUN_ALL_TESTS:
    test_results = []
    for number, question in enumerate(TEST_QUESTIONS, 1):
        print(f'Running Q{number}: {question}')
        result = run_agent(question)
        result['id'] = f'q{number:02d}'
        test_results.append(result)
        RESULTS_PATH.write_text(json.dumps(test_results, indent=2, ensure_ascii=False, default=str), encoding='utf-8')
else:
    test_results = []
    print('Set RUN_ALL_TESTS = True to run the five test questions.')

## 8. Review results, cost, and tool traces

Summary tables show how many times the agent called tools, how much it cost, and how long it took. The full JSON has the evidence returned by every tool.

In [ ]:
if test_results:
    summary = pd.DataFrame([{
        'id': item['id'], 'question': item['question'],
        'iterations': item['iterations'], 'tool_calls': item['tool_calls'],
        'tools_used': ', '.join(item['tools_used']),
        'input_tokens': item['input_tokens'], 'output_tokens': item['output_tokens'],
        'embedding_tokens': item['embedding_tokens'],
        'estimated_cost_usd': item['estimated_cost_usd'],
        'seconds': item['seconds'],
    } for item in test_results])
    summary.to_csv(REPORTS / f'agent_summary_{EMBEDDING_PROVIDER}.csv', index=False)
    display(summary)
    print('Total estimated USD:', round(summary['estimated_cost_usd'].sum(), 5))
else:
    print('No saved test results yet.')

In [ ]:
QUESTION_ID = 'q01'
selected = next((item for item in test_results if item['id'] == QUESTION_ID), None)
if selected:
    print('QUESTION:', selected['question'])
    print('\nANSWER:\n', selected['answer'])
    for event in selected['trace']:
        print('\nITERATION', event['iteration'], '|', event['action'])
        if 'arguments' in event:
            print('Arguments:', event['arguments'])
        if 'observation' in event:
            observation = event['observation']
            # Print source titles, not every full passage.
            for passage in observation.get('passages', []):
                print('-', passage['chunk_id'], source_title(passage))
            if observation.get('error'):
                print('Error:', observation['error'])
else:
    print('Run the five questions and choose a valid QUESTION_ID.')

## 9. Manual answer review

In [ ]:
REVIEW_PATH = REPORTS / f'agent_review_{EMBEDDING_PROVIDER}.csv'
if test_results and not REVIEW_PATH.exists():
    pd.DataFrame([{
        'id': item['id'], 'question': item['question'],
        'answer_quality': '',  # good / partial / poor
        'citations_correct': '',  # yes / partial / no
        'notes': '',
    } for item in test_results]).to_csv(REVIEW_PATH, index=False)

if REVIEW_PATH.exists():
    review = pd.read_csv(REVIEW_PATH, keep_default_na=False)
    display(review)
    print('Update these ratings after checking the original bank filings:', REVIEW_PATH)
else:
    print('Run the five questions to create the review sheet.')

Historical results and source provenance are retained in the evaluation reports. Human review ratings remain incomplete. Comparisons requiring 2023 are outside the 2024-2025 corpus.